# Demo G: SGLang Engine Benchmarks

**Platform:** Lightning.ai Studio (A100/L4 GPU)

**Goal:** Benchmark SGLang with the same metrics as Demo F (vLLM), then prove
RadixAttention's advantage for partial prefix matching.

## Same 4 metrics as Demo F:
| Metric | What it measures |
|---|---|
| Throughput (tok/s) | Aggregate tokens/second across concurrent users |
| TTFT (ms) | Time to first token |
| ITL (ms) | Inter-token latency |
| KV cache vs context | Memory growth with prompt length |

## Plus SGLang-specific:
| Test | What it proves |
|---|---|
| Exact prefix (shared system prompt) | Both engines cache well (tie) |
| Partial prefix (multi-turn, shared 3 turns) | SGLang wins (radix tree finds longest match) |

## Why SGLang?
vLLM hashes fixed-size blocks. If prompts differ at ANY point, hash breaks, no reuse.
SGLang builds a radix tree: finds the LONGEST common prefix token by token.
For agents, multi-turn chat, and RAG, this is the difference between cache hit and miss.

## Setup:
1. Same Lightning.ai Studio as Demo F
2. `pip install 'sglang[all]'` (once in terminal)
3. `pip install "scipy>=1.13" --no-deps` 
3. Kill vLLM, start SGLang server


In [1]:
# --- Setup: Same utilities as Demo F ---
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'openai', 'matplotlib>=3.9', 'scikit-learn>=1.5', 'scipy>=1.13', 'pandas>=2.2'])

import torch, time, requests, re, threading, gc
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─── Config ───
MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
SERVER_URL = f'http://localhost:{PORT}'
BASE_URL = f'{SERVER_URL}/v1'
N_REQUESTS = 10
N_TOKENS = 50

client = OpenAI(base_url=BASE_URL, api_key='unused')

# Same test prompts as Demo F
PROMPTS = [
    'Explain the concept of attention mechanisms in transformers',
    'What are the key differences between CPU and GPU architectures',
    'Describe how PagedAttention works in vLLM',
    'What is the roofline model for GPU performance analysis',
    'Explain continuous batching in LLM serving systems',
    'How does speculative decoding accelerate inference',
    'What are the tradeoffs of KV cache quantization',
    'Describe the prefill vs decode phases of LLM inference',
    'How do mixture of experts models reduce compute costs',
    'What is tensor parallelism and when should you use it',
]

# Prefix test data
SYSTEM_PREFIX = (
    'You are an expert systems architect specializing in distributed ML infrastructure. '
    'You have deep knowledge of GPU memory hierarchies, CUDA programming, network topologies, '
    'and large-scale model serving. Answer questions precisely and technically.'
)  # ~80 tokens

EXACT_QUESTIONS = [
    'How should I handle GPU OOM errors in production?',
    'What monitoring metrics matter for LLM serving?',
    'Compare disaggregated prefill vs colocated serving.',
    'How does request scheduling affect tail latency?',
    'What causes throughput degradation under high concurrency?',
]

SHARED_TURNS = (
    'User: What is a transformer?\n'
    'Assistant: A transformer is a neural network architecture based on self-attention.\n\n'
    'User: How does attention work?\n'
    'Assistant: Attention computes weighted sums of value vectors using query-key similarity.\n\n'
    'User: What is the complexity?\n'
    'Assistant: Standard attention is O(n^2) in sequence length.\n\n'
)  # ~100 tokens shared

TURN4_QUERIES = [
    'User: How can we reduce this to linear complexity?',
    'User: What is FlashAttention approach?',
    'User: Does this apply to cross-attention too?',
    'User: How does KV cache help with this?',
    'User: What about sparse attention patterns?',
]

# ─── Utilities (same as Demo F) ───
def check_server():
    try:
        r = requests.get(f'{SERVER_URL}/health', timeout=3)
        if r.status_code == 200:
            print('SGLang server running.')
            return True
    except: pass
    print('SGLang server NOT running.')
    return False

def get_server_metrics():
    """Scrape /metrics endpoint."""
    r = requests.get(f'{SERVER_URL}/metrics')
    metrics = {}
    for line in r.text.split('\n'):
        if line and not line.startswith('#'):
            parts = line.split()
            if len(parts) >= 2:
                try: metrics[parts[0]] = float(parts[-1])
                except: pass
    return metrics

def warmup_server(n=5):
    for i in range(n):
        requests.post(f'{BASE_URL}/completions',
            json={'model': MODEL, 'prompt': f'Warmup {i}', 'max_tokens': 20})
    print(f'Warmup done ({n} requests).')

def benchmark_full(prompts, max_tokens=50, n_workers=10, label=''):
    """Full benchmark: throughput, TTFT, ITL."""
    def single_request(prompt):
        t0 = time.perf_counter()
        first_token_time = None
        last_token_time = None
        tokens = 0
        stream = client.completions.create(
            model=MODEL, prompt=prompt, max_tokens=max_tokens,
            temperature=0, stream=True
        )
        for chunk in stream:
            now = time.perf_counter()
            if first_token_time is None: first_token_time = now
            last_token_time = now
            tokens += 1
        ttft = (first_token_time - t0) if first_token_time else 0
        itl = (last_token_time - first_token_time) / (tokens - 1) if tokens > 1 else 0
        return {'ttft': ttft, 'itl': itl, 'tokens': tokens}

    t_wall = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = [pool.submit(single_request, p) for p in prompts]
        req_results = [f.result() for f in tqdm(as_completed(futures), total=len(prompts), desc=label)]
    wall_time = time.perf_counter() - t_wall

    total_tokens = sum(r['tokens'] for r in req_results)
    throughput = total_tokens / wall_time
    avg_ttft = np.mean([r['ttft'] for r in req_results])
    avg_itl = np.mean([r['itl'] for r in req_results])
    print(f'  [{label}] {throughput:.0f} tok/s | TTFT: {avg_ttft*1000:.0f}ms | ITL: {avg_itl*1000:.1f}ms')
    return {'throughput': throughput, 'ttft': avg_ttft, 'itl': avg_itl, 'total_tokens': total_tokens}

def measure_kv_vs_context(context_lengths=[256, 512, 1024, 2048, 4096], n_users=5, max_tokens=500):
    """KV cache % at different context lengths."""
    kv_data = []
    base_text = 'The quick brown fox jumps over the lazy dog. ' * 500
    for ctx_len in tqdm(context_lengths, desc='KV vs context'):
        prompt = base_text[:ctx_len * 4]
        prompts_ctx = [prompt + f' Question {i}:' for i in range(n_users)]
        peak = 0.0
        def send():
            with ThreadPoolExecutor(max_workers=n_users) as pool:
                futs = [pool.submit(requests.post, f'{BASE_URL}/completions',
                    json={'model': MODEL, 'prompt': p, 'max_tokens': max_tokens, 'temperature': 0}
                ) for p in prompts_ctx]
                [f.result() for f in futs]
        t = threading.Thread(target=send)
        t.start()
        time.sleep(0.3)
        for _ in range(20):
            m = get_server_metrics()
            for k, v in m.items():
                if 'cache_usage' in k or 'kv_cache' in k:
                    peak = max(peak, v * 100)
            time.sleep(0.5)
        t.join()
        kv_data.append((ctx_len, peak))
        print(f'  {ctx_len:>5} ctx -> {peak:.1f}% KV')
    return kv_data

all_results = {}
print('Setup complete.')


Setup complete.


## Experiment 1: SGLang Baseline (Same Test as Demo F)

Same 10 prompts, same concurrency, same metrics. Fair comparison to vLLM.
SGLang uses RadixAttention by default (prefix caching always on, radix tree based).

**Start SGLang server:**
```bash
python -m sglang.launch_server \
    --model-path mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000
```

Wait for "The server is fired up and ready to roll!"


In [ ]:
# --- SGLang Baseline: Same benchmark as Demo F ---
assert check_server()
warmup_server()

sglang_metrics = benchmark_full(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label='SGLang')

print('\nKV cache vs context:')
sglang_kv_context = measure_kv_vs_context()

all_results['SGLang'] = {**sglang_metrics, 'kv_context': sglang_kv_context}
print(f'\nSGLang: {sglang_metrics["throughput"]:.0f} tok/s | TTFT: {sglang_metrics["ttft"]*1000:.0f}ms | ITL: {sglang_metrics["itl"]*1000:.1f}ms')


## Experiment 2: Exact Prefix Test

5 requests sharing an IDENTICAL system prompt (~80 tokens).
Both vLLM and SGLang cache this well. This is the control case.

Sent sequentially to see cold (request 1) vs warm (requests 2-5) TTFT.
After request 1 computes the prefix, requests 2-5 should have lower TTFT.


In [ ]:
# --- Exact Prefix: Sequential cold/warm ---
print('Exact prefix (shared system prompt, sequential)...')
exact_prompts = [f'{SYSTEM_PREFIX} {q}' for q in EXACT_QUESTIONS]

exact_ttfts = []
for i, prompt in enumerate(exact_prompts):
    t0 = time.perf_counter()
    first_token = None
    stream = client.completions.create(model=MODEL, prompt=prompt, max_tokens=N_TOKENS, temperature=0, stream=True)
    for chunk in stream:
        if first_token is None: first_token = time.perf_counter()
    ttft_ms = (first_token - t0) * 1000 if first_token else 0
    exact_ttfts.append(ttft_ms)
    tag = 'COLD' if i == 0 else 'WARM'
    print(f'  [{tag}] {EXACT_QUESTIONS[i][:40]:<40} TTFT: {ttft_ms:.0f}ms')

exact_cold = exact_ttfts[0]
exact_warm = np.mean(exact_ttfts[1:])
print(f'\nExact prefix: Cold {exact_cold:.0f}ms -> Warm {exact_warm:.0f}ms ({exact_cold/exact_warm:.2f}x speedup)')

all_results['SGLang (exact)'] = {'cold_ttft_ms': exact_cold, 'warm_ttft_ms': exact_warm, 'speedup': exact_cold/exact_warm, 'per_request': exact_ttfts}


## Experiment 3: Partial Prefix Test (SGLang's Key Advantage)

5 requests sharing the first 3 conversation turns (~100 tokens) but with DIFFERENT 4th turns.

**This is where SGLang wins:**
- **vLLM (hash-based):** Full prompt hash differs because 4th turn is different. No cache reuse.
- **SGLang (radix tree):** Traverses token by token, finds longest match (first 3 turns). Reuses that KV.

After request 1 stores the shared prefix in the radix tree, requests 2-5 only need to
compute KV for their unique 4th turn. TTFT drops proportionally.


In [ ]:
# --- Partial Prefix: SGLang radix tree advantage ---
print('Partial prefix (shared 3 turns, different 4th, sequential)...')
partial_prompts = [f'{SHARED_TURNS}{q}' for q in TURN4_QUERIES]

partial_ttfts = []
for i, prompt in enumerate(partial_prompts):
    t0 = time.perf_counter()
    first_token = None
    stream = client.completions.create(model=MODEL, prompt=prompt, max_tokens=N_TOKENS, temperature=0, stream=True)
    for chunk in stream:
        if first_token is None: first_token = time.perf_counter()
    ttft_ms = (first_token - t0) * 1000 if first_token else 0
    partial_ttfts.append(ttft_ms)
    tag = 'COLD' if i == 0 else 'WARM'
    print(f'  [{tag}] {TURN4_QUERIES[i][:40]:<40} TTFT: {ttft_ms:.0f}ms')

partial_cold = partial_ttfts[0]
partial_warm = np.mean(partial_ttfts[1:])
print(f'\nPartial prefix: Cold {partial_cold:.0f}ms -> Warm {partial_warm:.0f}ms ({partial_cold/partial_warm:.2f}x speedup)')
print(f'\nRadix tree finds the shared 3-turn prefix and reuses it!')
print('vLLM would show ~1x here (hash of full prompt differs, no reuse).')

all_results['SGLang (partial)'] = {'cold_ttft_ms': partial_cold, 'warm_ttft_ms': partial_warm, 'speedup': partial_cold/partial_warm, 'per_request': partial_ttfts}


## Final Comparison

Three panels:
1. Per-request TTFT for exact prefix (both requests drop after first)
2. Per-request TTFT for partial prefix (SGLang drops, vLLM would stay flat)
3. Speedup summary (exact vs partial)


In [ ]:
# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Exact prefix per-request TTFT
ax = axes[0]
ax.plot(range(1, len(exact_ttfts)+1), exact_ttfts, 'o-', color='#dcfce7',
       markeredgecolor='#000', linewidth=2.5, markersize=8, markeredgewidth=1)
ax.axhline(y=exact_ttfts[0], color='#991b1b', linestyle='--', alpha=0.5, label='Cold baseline')
ax.set_xlabel('Request #', fontsize=11, fontweight='bold')
ax.set_ylabel('TTFT (ms)', fontsize=11, fontweight='bold')
ax.set_title('Exact Prefix: TTFT Drops After Req 1', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 2: Partial prefix per-request TTFT
ax = axes[1]
ax.plot(range(1, len(partial_ttfts)+1), partial_ttfts, 's-', color='#dcfce7',
       markeredgecolor='#000', linewidth=2.5, markersize=8, markeredgewidth=1, label='SGLang')
ax.axhline(y=partial_ttfts[0], color='#dbeafe', linestyle='--', linewidth=2, alpha=0.8, label='vLLM (no partial cache)')
ax.set_xlabel('Request #', fontsize=11, fontweight='bold')
ax.set_ylabel('TTFT (ms)', fontsize=11, fontweight='bold')
ax.set_title('Partial Prefix: SGLang Drops, vLLM Flat', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 3: Speedup comparison
ax = axes[2]
categories = ['Exact Prefix', 'Partial Prefix']
speedups = [all_results['SGLang (exact)']['speedup'], all_results['SGLang (partial)']['speedup']]
bars = ax.bar(categories, speedups, color=['#dbeafe', '#dcfce7'], edgecolor='#000', linewidth=1.2, width=0.5)
for bar, val in zip(bars, speedups):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}x',
           ha='center', fontsize=12, fontweight='bold', color='#1e293b')
ax.axhline(y=1, color='#64748b', linestyle='--', alpha=0.3)
ax.set_ylabel('TTFT Speedup (cold/warm)', fontsize=11, fontweight='bold')
ax.set_title('SGLang Prefix Caching Speedup', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Demo G: SGLang RadixAttention', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('demo_g_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print('\n' + '='*55)
print(f'{"Metric":<30} {"Value":>12}')
print('-'*55)
print(f'{"Throughput (tok/s)":<30} {sglang_metrics["throughput"]:>12.0f}')
print(f'{"TTFT (ms)":<30} {sglang_metrics["ttft"]*1000:>12.0f}')
print(f'{"ITL (ms)":<30} {sglang_metrics["itl"]*1000:>12.1f}')
print(f'{"Exact prefix speedup":<30} {all_results["SGLang (exact)"]["speedup"]:>12.2f}x')
print(f'{"Partial prefix speedup":<30} {all_results["SGLang (partial)"]["speedup"]:>12.2f}x')
print('='*55)


## Per-Request Cache Analysis

The key insight: look at TTFT for EACH request, not just the average.
- Request 1: always slow (nothing cached yet)
- Request 2+: fast IF the engine found a cache hit

On the partial prefix workload, vLLM stays slow for ALL requests
(hash doesn't match), while SGLang drops after request 1 (radix tree finds overlap).


In [ ]:
# --- Per-Request TTFT Detail (shows cache hit pattern) ---
fig_detail, (ax_exact, ax_partial) = plt.subplots(1, 2, figsize=(12, 4))

# Exact prefix: per-request TTFT (should drop after first)
if 'vllm_exact' in dir() and 'sgl_exact' in dir():
    vllm_exact_ttfts = [r['ttft_ms'] for r in vllm_exact]
    sgl_exact_ttfts = [r['ttft_ms'] for r in sgl_exact]
    x_req = range(1, len(vllm_exact_ttfts) + 1)
    ax_exact.plot(x_req, vllm_exact_ttfts, 'o-', color='#2563eb', label='vLLM', markersize=5)
    ax_exact.plot(x_req, sgl_exact_ttfts, 's-', color='#16a34a', label='SGLang', markersize=5)
    ax_exact.set_xlabel('Request #')
    ax_exact.set_ylabel('TTFT (ms)')
    ax_exact.set_title('Exact Prefix: Per-Request TTFT', fontweight='bold')
    ax_exact.legend()
    ax_exact.spines['top'].set_visible(False)
    ax_exact.spines['right'].set_visible(False)

# Partial prefix: per-request TTFT (SGLang should be lower after first)
if 'vllm_partial' in dir() and 'sgl_partial' in dir():
    vllm_partial_ttfts = [r['ttft_ms'] for r in vllm_partial]
    sgl_partial_ttfts = [r['ttft_ms'] for r in sgl_partial]
    x_req2 = range(1, len(vllm_partial_ttfts) + 1)
    ax_partial.plot(x_req2, vllm_partial_ttfts, 'o-', color='#2563eb', label='vLLM', markersize=5)
    ax_partial.plot(x_req2, sgl_partial_ttfts, 's-', color='#16a34a', label='SGLang', markersize=5)
    ax_partial.set_xlabel('Request #')
    ax_partial.set_ylabel('TTFT (ms)')
    ax_partial.set_title('Partial Prefix: Per-Request TTFT\n(SGLang reuses partial, vLLM cannot)', fontweight='bold')
    ax_partial.legend()
    ax_partial.spines['top'].set_visible(False)
    ax_partial.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Left: Both engines cache exact prefixes (request 2+ is fast).')
print('Right: Only SGLang caches partial prefixes (vLLM stays slow for all).')


In [ ]:
# === Comparison Chart: vLLM vs SGLang Prefix Caching ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: Exact prefix comparison ---
ax1 = axes[0]
categories_exact = ["Cold\n(no cache)", "Warm\n(cached)"]
vllm_exact_means = [np.mean(vllm_exact_cold)*1000, np.mean(vllm_exact_warm)*1000]
sglang_exact_means = [np.mean(sglang_exact_cold)*1000, np.mean(sglang_exact_warm)*1000]

x_exact = np.arange(len(categories_exact))  # Bar positions
width = 0.35  # Bar width

bars1 = ax1.bar(x_exact - width/2, vllm_exact_means, width, label="vLLM", color="#dbeafe", edgecolor="#000")
bars2 = ax1.bar(x_exact + width/2, sglang_exact_means, width, label="SGLang", color="#dcfce7", edgecolor="#000")

ax1.set_xlabel("Run Type")
ax1.set_ylabel("Mean TTFT (ms)")
ax1.set_title("Exact Prefix: Both Engines Cache Well")
ax1.set_xticks(x_exact)
ax1.set_xticklabels(categories_exact)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars1 + bars2:
    height = bar.get_height()
    ax1.annotate(f"{height:.0f}", xy=(bar.get_x() + bar.get_width()/2, height),
                 xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

# --- Right panel: Partial prefix comparison (the key insight) ---
ax2 = axes[1]
categories_partial = ["Cold\n(no cache)", "Warm\n(cached)"]
vllm_partial_means = [np.mean(vllm_partial_cold)*1000, np.mean(vllm_partial_warm)*1000]
sglang_partial_means = [np.mean(sglang_partial_cold)*1000, np.mean(sglang_partial_warm)*1000]

x_partial = np.arange(len(categories_partial))  # Bar positions

bars3 = ax2.bar(x_partial - width/2, vllm_partial_means, width, label="vLLM", color="#dbeafe", edgecolor="#000")
bars4 = ax2.bar(x_partial + width/2, sglang_partial_means, width, label="SGLang", color="#dcfce7", edgecolor="#000")

ax2.set_xlabel("Run Type")
ax2.set_ylabel("Mean TTFT (ms)")
ax2.set_title("Partial Prefix: SGLang Radix Tree Wins")
ax2.set_xticks(x_partial)
ax2.set_xticklabels(categories_partial)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars3 + bars4:
    height = bar.get_height()
    ax2.annotate(f"{height:.0f}", xy=(bar.get_x() + bar.get_width()/2, height),
                 xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.suptitle("Demo G: Prefix Caching — vLLM (Hash) vs SGLang (Radix Tree)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("demo_g_prefix_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: demo_g_prefix_comparison.png")


## Key Takeaways: When to Use SGLang over vLLM for Prefix Caching

| Workload | vLLM | SGLang | Winner |
|----------|------|--------|--------|
| Identical system prompts | ✅ Cached (hash match) | ✅ Cached (radix match) | Tie |
| Multi-turn with shared history | ❌ Miss (hash of full prompt differs) | ✅ Hit (longest prefix match) | **SGLang** |
| RAG with shared retrieved docs | ❌ Miss | ✅ Partial hit | **SGLang** |
| Completely unique prompts | ❌ No benefit | ❌ No benefit | Tie |

### The Core Insight

**vLLM** hashes fixed-size token blocks. If the full block sequence matches exactly, cache hit. Any divergence breaks the hash chain.

**SGLang** uses a **radix tree** (trie) that finds the longest matching prefix character-by-character (token-by-token). Partial matches still save compute for the matched portion.

### When SGLang's Radix Tree Matters Most

1. **Multi-turn agents**: Conversation history grows but early turns are shared across requests
2. **RAG pipelines**: Retrieved context partially overlaps between queries  
3. **Few-shot prompting**: Examples are shared but the final query differs
4. **Code completion**: File context is shared but cursor position changes

### When It Doesn't Matter

- Fixed system prompts with short user queries (both engines cache fine)
- Completely unique prompts (nothing to cache)
- Single-request workloads (no reuse opportunity)
